## 1. Check GPU and disk

# LTX-2.5 22B on Kaggle — Wan2GP + Distilled NVFP4

Designed for **Kaggle T4**. This notebook keeps ALL heavy model/checkpoint files (main diffusion model, text encoder, VAEs, connectors, upscalers, vocoder — everything Wan2GP downloads into its `ckpts/` folder) on the **`/tmp`** partition instead of `/kaggle/working`, by symlinking the whole `ckpts` folder into `/tmp` *before* Wan2GP downloads anything. Nothing is staged file-by-file anymore — whatever Wan2GP fetches lands in `/tmp` automatically.

**Target test:** LTX-2.5 22B Distilled NVFP4, 480p, ~5 seconds, **8 diffusion steps + 3 extra post-processing/upscaling steps**.


In [1]:
!nvidia-smi

import torch, os
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), 
          f"{torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")

Mon Oct  5 19:33:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone Wan2GP

In [2]:
%cd /kaggle/working
import os
if not os.path.exists('/kaggle/working/Wan2GP'):
    !git clone https://github.com/deepbeepmeep/Wan2GP.git
%cd /kaggle/working/Wan2GP

/kaggle/working
Cloning into 'Wan2GP'...
remote: Enumerating objects: 19672, done.
remote: Counting objects: 100% (337/337), done.
remote: Compressing objects: 100% (278/278), done.
remote: Total 19672 (delta 81), reused 70 (delta 59), pack-reused 19335 (from 5)
Receiving objects: 100% (19672/19672), 92.16 MiB | 31.02 MiB/s, done.
Resolving deltas: 100% (13068/13068), done.
/kaggle/working/Wan2GP


## 3. Install Wan2GP dependencies

In [3]:
%cd /kaggle/working/Wan2GP

!pip install -r requirements.txt

/kaggle/working/Wan2GP
Looking in indexes: https://pypi.org/simple, https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/ort-cuda-13-nightly/pypi/simple/
Ignoring rembg: markers 'platform_system == "Darwin"' don't match your environment
Ignoring onnxruntime-gpu: markers 'python_version < "3.11" and platform_system != "Darwin"' don't match your environment
Ignoring onnxruntime: markers 'platform_system == "Darwin"' don't match your environment
Ignoring insightface: markers 'sys_platform == "win32" and python_version == "3.10"' don't match your environment
Ignoring insightface: markers 'sys_platform == "win32" and python_version == "3.11"' don't match your environment
Ignoring insightface: markers 'sys_platform == "win32" and python_version == "3.12"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import os

file1 = '/kaggle/working/Wan2GP/shared/utils/audio_video.py'
file2 = '/kaggle/working/Wan2GP/shared/utils/video_decode.py'

def patch_ffmpeg_file(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            content = f.read()
        if "-fps_mode" in content:
            content = content.replace("-fps_mode", "-vsync")
            with open(file_path, 'w') as f:
                f.write(content)
            print(f"✅ Successfully patched {file_path}")
        else:
            print(f"ℹ️ '-fps_mode' not found or already patched in {file_path}")
    else:
        print(f"❌ File not found: {file_path}")

patch_ffmpeg_file(file1)
patch_ffmpeg_file(file2)


ℹ️ '-fps_mode' not found or already patched in /kaggle/working/Wan2GP/shared/utils/audio_video.py
✅ Successfully patched /kaggle/working/Wan2GP/shared/utils/video_decode.py


## 4. Redirect ALL checkpoint/model storage to /tmp

In [5]:
import os, shutil

# --- Redirect ALL heavy model/checkpoint storage to /tmp ---
# Wan2GP downloads every checkpoint (main diffusion model, text encoder,
# VAEs, connectors, upscalers, vocoder, etc.) into a relative "ckpts/" folder
# inside the repo. Rather than staging files one at a time, we point that
# whole folder at /tmp via a symlink, so every download Wan2GP performs
# lands on the /tmp partition automatically and none of it touches
# /kaggle/working.

TMP_ROOT = '/tmp/ltx2.5'
CKPT_TARGET = os.path.join(TMP_ROOT, 'ckpts')
WAN_CKPT_DIR = '/kaggle/working/Wan2GP/ckpts'

os.makedirs(CKPT_TARGET, exist_ok=True)

# If ckpts already exists as a real (non-symlink) folder with files in it
# (e.g. left over from a run before this cell existed), move its contents
# into /tmp first so nothing already downloaded is lost or re-downloaded.
if os.path.exists(WAN_CKPT_DIR) and not os.path.islink(WAN_CKPT_DIR):
    print(f"Found existing {WAN_CKPT_DIR} — moving its contents to {CKPT_TARGET} ...")
    for item in os.listdir(WAN_CKPT_DIR):
        src = os.path.join(WAN_CKPT_DIR, item)
        dst = os.path.join(CKPT_TARGET, item)
        if not os.path.exists(dst):
            shutil.move(src, dst)
    shutil.rmtree(WAN_CKPT_DIR)
elif os.path.islink(WAN_CKPT_DIR):
    os.remove(WAN_CKPT_DIR)  # stale symlink from an earlier run, rebuild it

os.symlink(CKPT_TARGET, WAN_CKPT_DIR)
print(f"🔗 {WAN_CKPT_DIR} -> {CKPT_TARGET}")

# Also send the Hugging Face / transformers / torch / pip caches to /tmp,
# in case anything downloads through those caches instead of straight
# into ckpts/.
os.environ['HF_HOME'] = os.path.join(TMP_ROOT, 'hf_cache')
os.environ['HUGGINGFACE_HUB_CACHE'] = os.path.join(TMP_ROOT, 'hf_cache', 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(TMP_ROOT, 'hf_cache', 'transformers')
os.environ['TORCH_HOME'] = os.path.join(TMP_ROOT, 'torch_cache')
os.environ['PIP_CACHE_DIR'] = os.path.join(TMP_ROOT, 'pip_cache')

!df -h /tmp /kaggle/working


🔗 /kaggle/working/Wan2GP/ckpts -> /tmp/ltx2.5/ckpts
Filesystem      Size  Used Avail Use% Mounted on
overlay         8.0T  7.0T  1.1T  87% /
/dev/loop2       20G  192M   20G   1% /kaggle/working


## 5. Patch WanGP: auto-clean `/tmp` when you switch models in the interface

WanGP itself has no "delete the previous model" feature — it only ever downloads a model's files if they're missing, and leaves everything else alone (confirmed by reading `wgp.py`'s `download_models()`). To get automatic cleanup, this cell patches that function in-place, once, right before launch.

**What the patch does:** every time WanGP is about to download/load a model (i.e. whenever you pick a different model in the interface and hit Generate), it now:
1. Looks up which files the *previously* active model needed (its main checkpoint + shared encoder/VAE/upscaler files, read straight from WanGP's own `defaults/*.json` model definitions).
2. Looks up which files the *newly selected* model needs the same way.
3. Deletes anything from `/tmp/ltx2.5/ckpts` that the old model needed but the new one doesn't, then lets WanGP's normal download flow fetch whatever the new model is missing — straight into `/tmp` via the symlink from cell 4.

**Why this is safe by design:** if you switch between two variants that share the same text encoder/VAE (e.g. two LTX-2.5 checkpoints), those shared multi-GB files are kept — only the model-specific weights get removed — so you're not stuck re-downloading a 12GB encoder on every click. Switch to a completely different architecture (e.g. LTX-2.5 → Wan2.2) and its now-unused shared files get cleaned up too, since nothing in the new model's requirements still points to them.

**Caveat:** this works against the current WanGP source. It's wrapped in a try/except, so if a future WanGP update renames the internal functions this relies on, it will just print a warning and skip cleanup for that switch rather than crash the app — it won't silently corrupt anything, but you may need to re-check this cell after a big WanGP update.


In [6]:
import re

WGP_PY = '/kaggle/working/Wan2GP/wgp.py'
MARKER = '# --- WANGP_TMP_AUTO_CLEANUP_PATCH ---'

with open(WGP_PY, 'r') as f:
    src = f.read()

if MARKER in src:
    print('ℹ️  wgp.py already patched, skipping.')
else:
    anchor = (
        "def download_models(model_filename = None, model_type= None, file_type = 0, "
        "submodel_no = 1, force_path = None, model_def = None, gen=None, show_filename=True):\n"
        "    def computeList(filename):"
    )
    assert anchor in src, 'Could not find download_models() in this version of wgp.py — patch aborted, nothing changed.'

    injected = MARKER + "\n" + '''
def _tmp_switch_state_path():
    root = os.environ.get("WANGP_TMP_CKPT_ROOT", "/tmp/ltx2.5")
    return os.path.join(root, "_active_model.json")

def _tmp_resolve_needed_files(model_type):
    model_def = get_model_def(model_type)
    if model_def is None:
        return set()
    needed = set()
    lora_dir = get_lora_dir(model_type)

    for url in (model_def.get("URLs", []) or []):
        if not url:
            continue
        path = fl.get_local_model_filename(url, extra_paths=None)
        if path is None:
            path = fl.get_smart_download_location(url, force_path=None)
        if path:
            needed.add(os.path.abspath(path))

    for prop in ["preload_URLs", "VAE_URLs"]:
        vals = get_model_recursive_prop(model_type, prop, return_list=True, model_def=model_def)
        if isinstance(vals, str):
            vals = [vals]
        for url in (vals or []):
            if not url:
                continue
            path = fl.get_local_model_filename(url, lora_dir=lora_dir)
            if path is None:
                path = fl.get_download_location(url, lora_dir=lora_dir)
            if path:
                needed.add(os.path.abspath(path))
    return needed

def _tmp_ckpts_cleanup_on_switch(model_type):
    if not model_type:
        return
    try:
        state_file = _tmp_switch_state_path()
        prev_type = None
        if os.path.isfile(state_file):
            try:
                with open(state_file) as f:
                    prev_type = json.load(f).get("model_type")
            except Exception:
                prev_type = None
        if prev_type and prev_type != model_type:
            old_needed = _tmp_resolve_needed_files(prev_type)
            new_needed = _tmp_resolve_needed_files(model_type)
            freed = 0
            for path in (old_needed - new_needed):
                if os.path.isfile(path):
                    try:
                        size = os.path.getsize(path)
                        os.remove(path)
                        freed += size
                        print(f"[tmp-cleanup] removed {path}")
                    except Exception as e:
                        print(f"[tmp-cleanup] could not remove {path}: {e}")
            if freed:
                print(f"[tmp-cleanup] freed {freed/1e9:.1f} GB switching '{prev_type}' -> '{model_type}'")
        os.makedirs(os.path.dirname(state_file), exist_ok=True)
        with open(state_file, "w") as f:
            json.dump({"model_type": model_type}, f)
    except Exception as e:
        print(f"[tmp-cleanup] skipped due to error: {e}")
''' + "\n"

    new_def = (
        "def download_models(model_filename = None, model_type= None, file_type = 0, "
        "submodel_no = 1, force_path = None, model_def = None, gen=None, show_filename=True):\n"
        "    _tmp_ckpts_cleanup_on_switch(model_type)\n"
        "    def computeList(filename):"
    )

    patched = src.replace(anchor, injected + "\n" + new_def, 1)
    with open(WGP_PY, 'w') as f:
        f.write(patched)
    print('✅ wgp.py patched: models are now diffed and auto-cleaned from /tmp on every model switch.')


✅ wgp.py patched: models are now diffed and auto-cleaned from /tmp on every model switch.


## 6. Launch Wan2GP

In [ ]:
%cd /kaggle/working/Wan2GP
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

!python wgp.py --share --vae-config 3


/kaggle/working/Wan2GP
[VRAM] MMGP Optimized VRAM Allocator, spilling into RAM when VRAM is full
/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Switching to FP16 models when possible as GPU architecture doesn't support optimed BF16 Kernels
[INT8] Backend: Comfy Kitchen CUDA (setting: auto).
Loaded plugin: Motion Designer (from motion_designer)
WanGP web authentication disabled.
Deepy Web app: http://localhost:7860/deepy/
* Running on local URL:  http://localhost:7860
* Running on public URL: https://fcbbf11f41ed543754.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/ro

## 7. LTX-2.5 Checkpoint Paths

With the redirect cell above in place, **everything** Wan2GP downloads lives under:

```text
/tmp/ltx2.5/ckpts/
```

which `/kaggle/working/Wan2GP/ckpts` is symlinked to. That includes:

- Main checkpoint: `ltx-2.5-22b-distilled_diffusion_model_nvfp4.safetensors`
- Text encoder: `gemma4-12b-ltx-v1/gemma4-12b-ltx-v1_int8_convrot.safetensors`
- Video VAE, Diffusion Video VAE, Audio VAE
- Vocoder
- Text embedding projection
- Video/audio embedding connectors
- Spatial and temporal upscalers

None of this touches `/kaggle/working` — Wan2GP thinks it's downloading into its own `ckpts/` folder, but that folder is really a symlink into `/tmp`.


## 8. Suggested First-Generation Settings

For the first LTX-2.5 test:

* Model: **LTX-2.5 22B Distilled NVFP4**
* Steps: **8**
* Resolution: **480p**
* Duration: **~5 seconds**
* Batch size: **1**
* Keep other acceleration and quality options at their defaults for the first test.

### Expected Generation Time on Kaggle T4

With this setup, generation can be quite slow because the T4 does not have optimized NVFP4 kernels and therefore uses the fallback implementation.

As a rough estimate:

* **5-second video:** around **15 minutes**
* **10-second video:** around **30 minutes**

Actual generation time may vary depending on resolution, GPU availability, settings, and additional processing stages.

### What to Expect on a Kaggle T4

The NVFP4 checkpoint can run, but the T4 may show:

NVFP4: kernels unavailable; using fallback.
NVFP4: linear fallback

This is expected when optimized NVFP4 kernels are unavailable. The model can still generate successfully, but the fallback makes generation considerably slower.

The **8-step diffusion pass** is the main generation. Wan2GP may also perform separate decoding, upscaling, or post-processing afterward, which can show additional progress bars such as **3/3**.

**Do not interpret those additional stages as extra diffusion steps.**